# Machine Learning on "Belgian motor third-part liability dataset" from a rda dataset

### DATA COLLECTION
Install `pyreadr` library, which is capable of reading R data files (.rda), as it is not part of the standard Python distribution. Then download and load the `.rda` file from `https://github.com/dutangc/CASdatasets/blob/master/data/beMTPL16.rda`, extract the DataFrame, and display its head.

In [1]:
!pip install pyreadr
import requests
import os
import pyreadr
import pandas as pd
import numpy as np

raw_url = 'https://github.com/dutangc/CASdatasets/raw/master/data/beMTPL16.rda'
file_name = os.path.basename(raw_url)
print(f"Downloading {file_name}...")

try:
    if not os.path.exists(file_name):
        response = requests.get(raw_url)
        response.raise_for_status()
        with open(file_name, 'wb') as f:
            f.write(response.content)
        print("Downloaded successfully")
    else:
        print("Using cached file")

    result = pyreadr.read_r(file_name)
    df_beMTPL16 = result['beMTPL16']
    print(f"Data loaded: {df_beMTPL16.shape[0]} rows, {df_beMTPL16.shape[1]} columns")
except Exception as e:
    print(f"Error: {e}")


Using cached file
Using cached file
Data loaded: 70791 rows, 19 columns
Data loaded: 70791 rows, 19 columns


In [2]:
display(df_beMTPL16.head())

,insurance_contract,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,vehicle_brand,vehicle_model,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_time,claim_responsibility_rate,driving_training_label,signal
0,C1,1,0.386301,1945,10,9,40,MERCEDES,ME-1245,30000,75,983732,2,0,0,00:00,0,No,0
1,C2,1,0.493151,1941,4,25,24,VOLKSWAGEN,VO-2461,30000,55,510562,8,0,0,07:45,0,No,0
2,C3,1,0.290411,1944,0,2,39,AUDI,AU-967,30000,120,1934768,10,0,0,00:00,0,No,0
3,C4,1,0.336986,1948,1,14,37,LANCIA,LA-2346,30000,51,536755,13,0,0,18:50,0,No,0
4,C5,1,0.219178,1928,3,7,59,CITROEN,CI-1258,30000,54,446725,14,0,0,00:00,100,No,0


## 1. Data Validation
## Schema
| Column Name                                 | Type     | Description                                                         |
|---------------------------------------------|----------|---------------------------------------------------------------------|
| insurance_contract                           | numeric  | Unique identifier for the contract                                 |
| policy_year                                  | numeric  | Year of study/observation for the insured person                   |
| exposure                                     | numeric  | Exposure duration in years                                          |
| insured_year_birth                           | numeric  | Insured's year of birth                                             |
| vehicle_age                                  | numeric  | Age of the vehicle in years                                         |
| policy_holder_age                            | numeric  | Seniority of the insured at the insurance agency                    |
| driver_license_age                           | numeric  | Age of the driver's licence                                         |
| vehicle_brand                                | factor   | Brand of the vehicle                                                |
| vehicle_model                                | factor   | Model of the vehicle                                                |
| mileage                                      | numeric  | Mileage of the vehicle                                              |
| vehicle_power                                | numeric  | Power value of the vehicle                                          |
| catalog_value                                | numeric  | Catalog value of the vehicle                                        |
| claim_value                                  | numeric  | Value of the claim                                                  |
| number_of_liability_claims                   | numeric  | Number of liability claims                                          |
| number_of_bodily_injury_liability_claims     | numeric  | Number of bodily injury liability claims                            |
| claim_time                                   | factor   | Time (within a day) of the accident                                 |
| claim_responsibility_rate                    | numeric  | Responsibility rate (0–100%)                                        |
| driving_training_label                       | factor   | Indicator for driving training program                              |
| signal                                       | numeric  | Warning indicator (1 = warning, 0 = no warning)                     |


Perform initial data validation by checking data types, non-null counts, and summary statistics of the `df_beMTPL16` DataFrame.


**Info()**: The `info()` method provides a concise summary of a DataFrame, including the number of entries, the number of columns, the data type of each column, the number of non-null values, and memory usage. This is crucial for identifying missing data and incorrect data types right away.

**Describe()**: The `describe()` method generates descriptive statistics that summarize the central tendency, dispersion, and shape of a dataset's distribution, excluding `NaN` values. This helps in understanding the distribution and potential outliers in numerical columns.

In [3]:
print("Dataset Overview")
df_beMTPL16.info()

print("\nNull Values:")
print(df_beMTPL16.isnull().sum())

cat_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype == 'category']
num_col = [col for col in df_beMTPL16.columns if df_beMTPL16[col].dtype != 'category']

print(f'\nColumns: {len(cat_col)} categorical, {len(num_col)} numerical')

zero_counts = {col: (df_beMTPL16[col] == 0).sum() for col in num_col}
zero_df = pd.DataFrame([(col, count, f"{count/len(df_beMTPL16)*100:.2f}%") 
                         for col, count in zero_counts.items() if count > 0],
                       columns=['Column', 'Zero Count', 'Percentage'])
print("\nZero Values:")
display(zero_df.sort_values(by='Zero Count', ascending=False))

print("\nStatistics:")
display(df_beMTPL16.describe())


Dataset Overview
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70791 entries, 0 to 70790
Data columns (total 19 columns):
 #   Column                                    Non-Null Count  Dtype   
---  ------                                    --------------  -----   
 0   insurance_contract                        70791 non-null  category
 1   policy_year                               70791 non-null  int32   
 2   exposure                                  70791 non-null  float64 
 3   insured_birth_year                        70791 non-null  int32   
 4   vehicle_age                               70791 non-null  int32   
 5   policy_holder_age                         70791 non-null  int32   
 6   driver_license_age                        70791 non-null  int32   
 7   vehicle_brand                             70791 non-null  category
 8   vehicle_model                             70791 non-null  category
 9   mileage                                   70791 non-null  int32   
 10  vehic

,Column,Zero Count,Percentage
7,signal,70746,99.94%
5,number_of_bodily_injury_liability_claims,69381,98.01%
4,number_of_liability_claims,46080,65.09%
6,claim_responsibility_rate,36017,50.88%
3,catalog_value,21844,30.86%
0,vehicle_age,3304,4.67%
1,policy_holder_age,2331,3.29%
2,driver_license_age,3,0.00%



Statistics:


,policy_year,exposure,insured_birth_year,vehicle_age,policy_holder_age,driver_license_age,mileage,vehicle_power,catalog_value,claim_value,number_of_liability_claims,number_of_bodily_injury_liability_claims,claim_responsibility_rate,signal
count,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000,7.079100e+04,70791.000000,70791.000000,70791.000000,70791.000000,70791.000000
mean,2.503934,0.437601,1941.394231,6.163849,9.651679,38.196875,28327.824158,77.962382,5.823853e+05,80780.786371,0.349070,0.019918,48.424362,0.000636
std,1.108830,0.180683,6.960452,4.803108,6.789322,9.549636,5711.127945,29.633158,5.460070e+05,45999.887220,0.476679,0.139719,49.628070,0.025205
min,1.000000,0.200000,1911.000000,0.000000,0.000000,0.000000,2500.000000,30.000000,0.000000e+00,2.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,0.287671,1936.000000,2.000000,4.000000,34.000000,30000.000000,55.000000,0.000000e+00,41491.500000,0.000000,0.000000,0.000000,0.000000
50%,3.000000,0.400000,1943.000000,5.000000,9.000000,41.000000,30000.000000,74.000000,5.506000e+05,82430.000000,0.000000,0.000000,0.000000,0.000000
75%,3.000000,0.556164,1947.000000,9.000000,14.000000,43.000000,30000.000000,92.000000,8.677665e+05,120472.000000,1.000000,0.000000,100.000000,0.000000
max,4.000000,1.000000,1952.000000,60.000000,30.000000,60.000000,30000.000000,487.000000,7.234528e+06,169694.000000,1.000000,1.000000,100.000000,1.000000


#### 1. Columns to Drop

These columns should be removed from your dataset before training for the reasons specified.

> **Note on Column Names:** The GitHub documentation refers to `insured_year_birth`, but the actual R dataset uses `insured_birth_year`. We use the real data column name (`insured_birth_year`) in this notebook.

| Column Name | Reason to Drop |
| :--- | :--- |
| `insurance_contract` | Unique identifier, no predictive value. |
| `insured_birth_year` | Redundant (you already have `policy_holder_age`). |
| `exposure` | Irrelevant for a severity-only model (it's for frequency). |
| `number_of_liability_claims` | **Data Leakage** (This is an *outcome* of the claim). |
| `number_of_bodily_injury_liability_claims` | **Data Leakage** (This is a *component* of the claim value). |
| `claim_responsibility_rate` | **Data Leakage** (This is determined *during* claim adjustment). |
| `signal` | **Near-Zero-Variance** (99.94% zero, no predictive power). |
| `driving_training_label` | **Low Real-World Value:** Sparse indicator with limited predictive power; excluded for model simplicity. |



#### 2. Target Label & Features to Keep

This is the set of columns you will use to build your model.

#### Target Label (Y)

This is the single column you are trying to predict.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `claim_value` | **Target (Label)** | This is the value your model will predict. Remember to filter your data for `claim_value > 0` before training. |

#### Features (X)

These are the columns your model will use to make its predictions.

| Column Name | Role | Notes |
| :--- | :--- | :--- |
| `policy_year` | Feature | Good for tracking trends/inflation over time. |
| `vehicle_age` | Feature | Keep as-is. The zeros (4.67%) likely mean "new car." |
| `policy_holder_age` | Feature | **Must clean:** The zeros (3.29%) are missing data and should be imputed. |
| `driver_license_age` | Feature | **Must clean:** The 3 zero-rows are missing data. |
| `vehicle_brand` | Feature | Categorical. Will require encoding. |
| `vehicle_model` | Feature | Categorical. May have many values; consider grouping rare models. |
| `mileage` | Feature | Numeric feature. |
| `vehicle_power` | Feature | Numeric feature. |
| `catalog_value` | Feature | **Must clean:** The zeros (30.86%) are missing data and must be imputed. |
| `claim_time` | Feature | Categorical (time of day). Will require encoding. |

In [4]:
print("Data Cleaning")

cols_to_drop = [
    'insurance_contract', 'insured_birth_year', 'exposure',
    'number_of_liability_claims', 'number_of_bodily_injury_liability_claims',
    'signal', 'claim_responsibility_rate', 'driving_training_label'
]

df_clean = df_beMTPL16.drop(columns=cols_to_drop, errors='ignore').copy()
print(f"After dropping columns: {df_clean.shape}")

df_clean = df_clean[df_clean['driver_license_age'] > 0]
print(f"After removing driver_license_age == 0: {df_clean.shape}")

df_model_data = df_clean[df_clean['claim_value'] > 0].copy()
print(f"After filtering claim_value > 0: {df_model_data.shape}")

# Smart Imputation for catalog_value using Random Forest
print("\nSmart Imputation for catalog_value...")
from sklearn.ensemble import RandomForestRegressor as RFR_Imputer

# Identify rows with missing catalog_value (zeros)
catalog_missing_mask = df_model_data['catalog_value'] == 0
catalog_present_mask = ~catalog_missing_mask

# If there are missing values, train a model to predict them
if catalog_missing_mask.sum() > 0:
    # Features for imputation: vehicle characteristics
    imputation_features = ['vehicle_brand', 'vehicle_power', 'vehicle_age']
    
    # Prepare training data (rows where catalog_value is not zero)
    X_catalog_train = df_model_data.loc[catalog_present_mask, imputation_features].copy()
    y_catalog_train = df_model_data.loc[catalog_present_mask, 'catalog_value']
    
    # Convert categorical to numeric for RF imputer
    X_catalog_train['vehicle_brand'] = X_catalog_train['vehicle_brand'].astype('category').cat.codes
    
    # Train RF imputer
    rf_imputer = RFR_Imputer(n_estimators=100, random_state=42, n_jobs=-1)
    rf_imputer.fit(X_catalog_train, y_catalog_train)
    
    # Predict missing values
    X_catalog_missing = df_model_data.loc[catalog_missing_mask, imputation_features].copy()
    X_catalog_missing['vehicle_brand'] = X_catalog_missing['vehicle_brand'].astype('category').cat.codes
    predicted_values = rf_imputer.predict(X_catalog_missing)
    
    # Fill in missing values
    df_model_data.loc[catalog_missing_mask, 'catalog_value'] = predicted_values
    print(f"Imputed {catalog_missing_mask.sum()} missing catalog_value entries using Random Forest")
else:
    print("No missing catalog_value entries found")

# Impute policy_holder_age with median (zeros represent missing data)
median_policy_holder_age = df_model_data[df_model_data['policy_holder_age'] > 0]['policy_holder_age'].median()
df_model_data['policy_holder_age'] = df_model_data['policy_holder_age'].replace(0, median_policy_holder_age)
print(f"Imputed policy_holder_age with median: {median_policy_holder_age:.2f}")

Data Cleaning
After dropping columns: (70791, 11)
After removing driver_license_age == 0: (70788, 11)
After filtering claim_value > 0: (70788, 11)

Smart Imputation for catalog_value...
After dropping columns: (70791, 11)
After removing driver_license_age == 0: (70788, 11)
After filtering claim_value > 0: (70788, 11)

Smart Imputation for catalog_value...
Imputed 21843 missing catalog_value entries using Random Forest
Imputed policy_holder_age with median: 9.00
Imputed 21843 missing catalog_value entries using Random Forest
Imputed policy_holder_age with median: 9.00


/var/folders/h8/ybzdwy6x03l4xpqzfc0k7kpr0000gn/T/ipykernel_3123/2873108735.py:48: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[314971.58531183 764724.46989659 665613.51980303 ... 647275.75601709
 866550.58066541 467040.5048824 ]' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df_model_data.loc[catalog_missing_mask, 'catalog_value'] = predicted_values


## 2. Feature Engineering

Create the feature matrix (X) and target vector (y) for modeling. The `features` list is easily configurable to experiment with different feature combinations. The target variable (`claim_value`) is log-transformed to normalize the distribution and improve model performance.


In [5]:
from sklearn.model_selection import train_test_split

print("Feature Engineering")

# Feature configuration - easy to modify for experiments
features = [
    'policy_year', 'vehicle_age', 'policy_holder_age', 'driver_license_age',
    'vehicle_brand', 'vehicle_model', 'mileage', 'vehicle_power',
    'catalog_value', 'claim_time'
]
target = 'claim_value'

X = df_model_data[features]
y = np.log1p(df_model_data[target])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")


Feature Engineering
Train: 56630 rows, Test: 14158 rows


## 3. Preprocessing Pipeline

Build a preprocessing pipeline using scikit-learn's `ColumnTransformer` to:
- **Numeric features**: Apply StandardScaler for normalization
- **Categorical features**: Apply OneHotEncoder for categorical encoding

This pipeline ensures consistent transformation of both training and test data. Modify the `numeric_features` and `categorical_features` lists to match your feature engineering choices.


In [6]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

print("Preprocessing Pipeline")

# Preprocessing configuration - modify based on features list
numeric_features = [
    'policy_year', 'vehicle_age', 'policy_holder_age',
    'driver_license_age', 'mileage', 'vehicle_power', 'catalog_value'
]
categorical_features = ['vehicle_brand', 'vehicle_model', 'claim_time']

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=0.01, sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='drop'
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

processed_feature_names = preprocessor.get_feature_names_out()
print(f"Processed features: {X_train_processed.shape[1]}")

# Actuarial XGBoost: Create a separate pipeline with Target Encoding
print("\n\nActuarial XGBoost Preprocessing Pipeline (with Target Encoding)")
import subprocess
import sys

# Install category_encoders if not already installed
try:
    import category_encoders
    print("category_encoders is already installed")
except ImportError:
    print("Installing category_encoders...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "category_encoders", "-q"])
    import category_encoders

from category_encoders import TargetEncoder

# Prepare y values for target encoding (fit on training data)
y_train_original = np.expm1(y_train)  # Convert back to original scale for target encoding

numeric_transformer_actuarial = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer_actuarial = ColumnTransformer(
    transformers=[
        ('target_high', TargetEncoder(), ['vehicle_brand', 'vehicle_model']),
        ('onehot_low', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['claim_time'])
    ],
    remainder='drop'
)

preprocessor_actuarial = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer_actuarial, numeric_features),
        ('cat', categorical_transformer_actuarial, categorical_features)
    ],
    remainder='drop'
)

# Fit target encoder on training data with original-scale targets
preprocessor_actuarial.fit(X_train, y_train_original)
X_train_actuarial = preprocessor_actuarial.transform(X_train)
X_test_actuarial = preprocessor_actuarial.transform(X_test)

print(f"Actuarial processed features: {X_train_actuarial.shape[1]}")
print("Target encoding applied to vehicle_brand and vehicle_model")

Preprocessing Pipeline
Processed features: 67


Actuarial XGBoost Preprocessing Pipeline (with Target Encoding)
category_encoders is already installed
category_encoders is already installed
Actuarial processed features: 1005
Target encoding applied to vehicle_brand and vehicle_model
Actuarial processed features: 1005
Target encoding applied to vehicle_brand and vehicle_model


## 4. Model Training & Evaluation

Train multiple regression models to predict claim severity:
- **Linear Regression**: Simple baseline for comparison
- **XGBoost (basic)**: Gradient boosting with early stopping
- **Random Forest**: Ensemble method for robustness
- **XGBoost (tuned)**: Optimized hyperparameters using RandomizedSearchCV

Each model is trained, evaluated independently, and results are compared on RMSE in log scale and original EUR scale.

### 4.1 Linear Regression Baseline

Train a simple linear regression model as a baseline. This provides a straightforward prediction and helps establish a performance benchmark against more complex models.

## MLflow Setup & Configuration

Install and configure MLflow for experiment tracking, parameter logging, and model management.


In [7]:
!pip install mlflow -q

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Set tracking URI (local file system)
mlflow.set_tracking_uri("./mlruns")

# Create or set the experiment
experiment_name = "Belgian_MTPL_Severity"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
except mlflow.exceptions.MlflowException:
    # Experiment already exists
    experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

mlflow.set_experiment(experiment_name)

# Configure autologging: 
# - log_input_examples=False: Prevents pandas serialization warnings when logging training data
# - log_model_signatures=False: Prevents "logged without signature" warnings for models with transformed inputs
# These are appropriate for our use case since:
#   1. We're using preprocessed numpy arrays (not raw dataframes)
#   2. We manually log important metrics (rmse_eur, mape)
#   3. The model artifacts are still saved for deployment
mlflow.sklearn.autolog(log_input_examples=False, log_model_signatures=False)
mlflow.xgboost.autolog(log_input_examples=False, log_model_signatures=False)

print(f"MLflow configured successfully!")
print(f"Experiment: {experiment_name}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Run logs will be saved to ./mlruns directory")


/Users/charlesnanakwakye/HobbyApps/be-insurance-ai/.venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)


MLflow configured successfully!
Experiment: Belgian_MTPL_Severity
Tracking URI: ./mlruns
Run logs will be saved to ./mlruns directory


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Baseline_Linear_Regression"):
    model_lr = LinearRegression()
    model_lr.fit(X_train_processed, y_train)
    pred_lr = model_lr.predict(X_test_processed)
    test_rmse_lr = np.sqrt(mean_squared_error(y_test, pred_lr))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_lr_original = np.expm1(pred_lr)
    rmse_lr_eur = np.sqrt(mean_squared_error(y_test_original, pred_lr_original))
    mape_lr = mean_absolute_percentage_error(y_test_original, pred_lr_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_lr_eur)
    mlflow.log_metric("mape", mape_lr)
    
    print(f"Linear Regression RMSE (log scale): {test_rmse_lr:.6f}")
    print(f"  RMSE (EUR): €{rmse_lr_eur:.2f}")
    print(f"  MAPE: {mape_lr * 100:.2f}%")

2025/11/22 02:41:51 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:41:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:41:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Linear Regression RMSE (log scale): 0.504837
  RMSE (EUR): €24445.42
  MAPE: 59.66%


### 4.2 XGBoost (Basic Configuration)

Train XGBoost with standard hyperparameters and early stopping to prevent overfitting. Early stopping monitors validation performance and halts training when no improvement is observed.

In [9]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGBoost_Basic"):
    model_xgb = xgb.XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=5,
        early_stopping_rounds=50,
        random_state=42
    )
    model_xgb.fit(
        X_train_processed, y_train,
        eval_set=[(X_test_processed, y_test)],
        verbose=False
    )
    pred_xgb = model_xgb.predict(X_test_processed)
    test_rmse_xgb = np.sqrt(mean_squared_error(y_test, pred_xgb))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_xgb_original = np.expm1(pred_xgb)
    rmse_xgb_eur = np.sqrt(mean_squared_error(y_test_original, pred_xgb_original))
    mape_xgb = mean_absolute_percentage_error(y_test_original, pred_xgb_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_xgb_eur)
    mlflow.log_metric("mape", mape_xgb)
    
    print(f"XGBoost (Basic) RMSE (log scale): {test_rmse_xgb:.6f}")
    print(f"  RMSE (EUR): €{rmse_xgb_eur:.2f}")
    print(f"  MAPE: {mape_xgb * 100:.2f}%")

2025/11/22 02:41:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 02:41:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:41:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost (Basic) RMSE (log scale): 0.428026
  RMSE (EUR): €12301.53
  MAPE: 40.94%


### 4.3 Random Forest

Train a Random Forest model using 200 trees with controlled depth to balance bias and variance. This ensemble approach reduces overfitting and provides robust predictions.

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="RF_Ensemble"):
    model_rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    model_rf.fit(X_train_processed, y_train)
    pred_rf = model_rf.predict(X_test_processed)
    test_rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    pred_rf_original = np.expm1(pred_rf)
    rmse_rf_eur = np.sqrt(mean_squared_error(y_test_original, pred_rf_original))
    mape_rf = mean_absolute_percentage_error(y_test_original, pred_rf_original)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_rf_eur)
    mlflow.log_metric("mape", mape_rf)
    
    print(f"Random Forest RMSE (log scale): {test_rmse_rf:.6f}")
    print(f"  RMSE (EUR): €{rmse_rf_eur:.2f}")
    print(f"  MAPE: {mape_rf * 100:.2f}%")

2025/11/22 02:41:57 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:42:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:42:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest RMSE (log scale): 0.427920
  RMSE (EUR): €12309.59
  MAPE: 40.20%


### 4.4 XGBoost with Hyperparameter Tuning

Optimize XGBoost hyperparameters using RandomizedSearchCV with 5-fold cross-validation. This explores the hyperparameter space to find configurations that maximize model performance.

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="XGB_Tuned_RandomSearch"):
    param_dist = {
        'n_estimators': randint(200, 1500),
        'learning_rate': uniform(0.01, 0.1),
        'max_depth': randint(3, 8),
        'subsample': uniform(0.7, 0.3),
        'colsample_bytree': uniform(0.7, 0.3)
    }

    random_search = RandomizedSearchCV(
        xgb.XGBRegressor(random_state=42),
        param_distributions=param_dist,
        n_iter=50,
        cv=5,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    random_search.fit(X_train_processed, y_train)
    best_model_xgb = random_search.best_estimator_
    log_predictions_tuned = best_model_xgb.predict(X_test_processed)
    final_test_rmse = np.sqrt(mean_squared_error(y_test, log_predictions_tuned))
    
    # Convert to original scale for business metrics
    y_test_original = np.expm1(y_test)
    predictions_original = np.expm1(log_predictions_tuned)
    rmse_tuned_eur = np.sqrt(mean_squared_error(y_test_original, predictions_original))
    mape_tuned = mean_absolute_percentage_error(y_test_original, predictions_original)
    
    # Log best hyperparameters
    mlflow.log_params(random_search.best_params_)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_tuned_eur)
    mlflow.log_metric("mape", mape_tuned)

    print(f"XGBoost (Tuned) RMSE (log scale): {final_test_rmse:.6f}")
    print(f"  RMSE (EUR): €{rmse_tuned_eur:.2f}")
    print(f"  MAPE: {mape_tuned * 100:.2f}%")
    print(f"\nBest Hyperparameters:")
    for param, value in random_search.best_params_.items():
        print(f"  {param}: {value}")

2025/11/22 02:42:07 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/22 02:47:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:13 INFO mlflow.sklearn.utils: Logging the

XGBoost (Tuned) RMSE (log scale): 0.427424
  RMSE (EUR): €12336.02
  MAPE: 40.62%

Best Hyperparameters:
  colsample_bytree: 0.8023199053150775
  learning_rate: 0.02134735212405891
  max_depth: 4
  n_estimators: 462
  subsample: 0.8979952138102536


### 4.5 Actuarial XGBoost (Gamma Regression)

Train an XGBoost model specifically designed for insurance modeling using Gamma regression. This approach:
- Uses original claim values (not log-transformed) to better model right-skewed severity distributions
- Employs Gamma objective function which is appropriate for non-negative continuous data
- Uses Target Encoding for high-cardinality categorical variables (vehicle_brand, vehicle_model)
- Incorporates smart-imputed features from Random Forest predictions

In [12]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

with mlflow.start_run(run_name="Actuarial_XGB_Gamma"):
    # Critical: Use original claim values (not log-transformed) for Gamma regression
    y_train_original = np.expm1(y_train)
    y_test_original = np.expm1(y_test)

    # Train Actuarial XGBoost with Gamma regression
    model_xgb_actuarial = xgb.XGBRegressor(
        objective='reg:gamma',
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=50,
        random_state=42,
        n_jobs=-1
    )

    model_xgb_actuarial.fit(
        X_train_actuarial, y_train_original,
        eval_set=[(X_test_actuarial, y_test_original)],
        verbose=False
    )

    # Predict on test set
    pred_xgb_actuarial = model_xgb_actuarial.predict(X_test_actuarial)

    # Evaluate on original scale
    rmse_actuarial = np.sqrt(mean_squared_error(y_test_original, pred_xgb_actuarial))
    mape_actuarial = mean_absolute_percentage_error(y_test_original, pred_xgb_actuarial)
    
    # Log custom metrics
    mlflow.log_metric("rmse_eur", rmse_actuarial)
    mlflow.log_metric("mape", mape_actuarial)

    print(f"Actuarial XGBoost (Gamma Regression) Performance:")
    print(f"  RMSE (original scale, EUR): €{rmse_actuarial:.2f}")
    print(f"  MAPE: {mape_actuarial * 100:.2f}%")

2025/11/22 02:47:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 02:47:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/22 02:47:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Actuarial XGBoost (Gamma Regression) Performance:
  RMSE (original scale, EUR): €12113.84
  MAPE: 50.69%


## 5. Results Comparison, Evaluation & Detailed Predictions

Compare all models on both log scale (training metric) and original EUR scale (business metric). View detailed predictions on test claims to see how each model performs on individual cases.


In [13]:
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

y_test_original = np.expm1(y_test)
predictions_original = np.expm1(log_predictions_tuned)

print("=" * 80)
print("MODEL COMPARISON: ALL 5 MODELS")
print("=" * 80)
print("\nNote: Detailed metrics, hyperparameters, and model artifacts are now")
print("tracked in MLflow. View the dashboard with: mlflow ui")
print("=" * 80)

print("\nModel Performance (RMSE on original scale, EUR):")
print(f"  1. Linear Regression:              €{np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_lr))):.2f}")
print(f"  2. XGBoost (basic):                €{np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_xgb))):.2f}")
print(f"  3. Random Forest:                  €{np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_rf))):.2f}")
print(f"  4. XGBoost (tuned, log-normal):    €{rmse_tuned_eur:.2f}")
print(f"  5. Actuarial XGBoost (gamma):      €{rmse_actuarial:.2f}")

print("\nModel Performance (MAPE):")
mape_lr_final = mean_absolute_percentage_error(y_test_original, np.expm1(pred_lr))
mape_xgb_final = mean_absolute_percentage_error(y_test_original, np.expm1(pred_xgb))
mape_rf_final = mean_absolute_percentage_error(y_test_original, np.expm1(pred_rf))

print(f"  1. Linear Regression:              {mape_lr_final * 100:.2f}%")
print(f"  2. XGBoost (basic):                {mape_xgb_final * 100:.2f}%")
print(f"  3. Random Forest:                  {mape_rf_final * 100:.2f}%")
print(f"  4. XGBoost (tuned, log-normal):    {mape_tuned * 100:.2f}%")
print(f"  5. Actuarial XGBoost (gamma):      {mape_actuarial * 100:.2f}%")

print("\n" + "=" * 80)
print("BEST MODEL SUMMARY")
print("=" * 80)

# Find best model
models_rmse = {
    'Linear Regression': np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_lr))),
    'XGBoost (basic)': np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_xgb))),
    'Random Forest': np.sqrt(mean_squared_error(y_test_original, np.expm1(pred_rf))),
    'XGBoost (tuned)': rmse_tuned_eur,
    'Actuarial XGBoost': rmse_actuarial
}

best_model = min(models_rmse, key=models_rmse.get)
best_mape = [mape_lr_final, mape_xgb_final, mape_rf_final, mape_tuned, mape_actuarial][list(models_rmse.keys()).index(best_model)]

print(f"\nBest Model: {best_model}")
print(f"  RMSE: €{models_rmse[best_model]:.2f}")
print(f"  MAPE: {best_mape * 100:.2f}%")
print(f"  Average Claim Value: €{y_test_original.mean():.2f}")

print("\n" + "=" * 80)
print("DETAILED PREDICTIONS ON TEST SET (First 20 claims)")
print("=" * 80)

# Convert predictions back to original scale for all models
pred_lr_original = np.expm1(pred_lr)
pred_xgb_original = np.expm1(pred_xgb)
pred_rf_original = np.expm1(pred_rf)
pred_tuned_original = np.expm1(log_predictions_tuned)

# Create a comprehensive results dataframe
results_df = pd.DataFrame({
    'Actual': y_test_original,
    'Linear Regression': pred_lr_original,
    'XGBoost (Basic)': pred_xgb_original,
    'Random Forest': pred_rf_original,
    'XGBoost (Tuned)': pred_tuned_original,
    'Actuarial XGBoost': pred_xgb_actuarial
})

results_df = results_df.reset_index(drop=True)

print("\nShowing actual claim values vs predictions from all 5 models (in EUR):\n")
display(results_df.head(20).round(2))

print("\n" + "=" * 80)
print("MODEL PREDICTION ERRORS (First 20 claims)")
print("=" * 80)
print("\nAbsolute Error (EUR) for each model:\n")

errors_df = pd.DataFrame({
    'Actual': y_test_original,
    'Linear Regression Error': np.abs(pred_lr_original - y_test_original),
    'XGBoost (Basic) Error': np.abs(pred_xgb_original - y_test_original),
    'Random Forest Error': np.abs(pred_rf_original - y_test_original),
    'XGBoost (Tuned) Error': np.abs(pred_tuned_original - y_test_original),
    'Actuarial XGBoost Error': np.abs(pred_xgb_actuarial - y_test_original)
})

errors_df = errors_df.reset_index(drop=True)
display(errors_df.head(20).round(2))

print("\n" + "=" * 80)
print("MODEL ERROR STATISTICS")
print("=" * 80)

error_stats = pd.DataFrame({
    'Model': ['Linear Regression', 'XGBoost (Basic)', 'Random Forest', 'XGBoost (Tuned)', 'Actuarial XGBoost'],
    'Mean Absolute Error (EUR)': [
        np.abs(pred_lr_original - y_test_original).mean(),
        np.abs(pred_xgb_original - y_test_original).mean(),
        np.abs(pred_rf_original - y_test_original).mean(),
        np.abs(pred_tuned_original - y_test_original).mean(),
        np.abs(pred_xgb_actuarial - y_test_original).mean()
    ],
    'Median Absolute Error (EUR)': [
        np.median(np.abs(pred_lr_original - y_test_original)),
        np.median(np.abs(pred_xgb_original - y_test_original)),
        np.median(np.abs(pred_rf_original - y_test_original)),
        np.median(np.abs(pred_tuned_original - y_test_original)),
        np.median(np.abs(pred_xgb_actuarial - y_test_original))
    ],
    'Max Error (EUR)': [
        np.abs(pred_lr_original - y_test_original).max(),
        np.abs(pred_xgb_original - y_test_original).max(),
        np.abs(pred_rf_original - y_test_original).max(),
        np.abs(pred_tuned_original - y_test_original).max(),
        np.abs(pred_xgb_actuarial - y_test_original).max()
    ]
})

print("\n")
display(error_stats.round(2))

print("\n" + "=" * 80)
print("TO VIEW MLFLOW DASHBOARD:")
print("   Run in terminal: mlflow ui")
print("   Then open: http://localhost:5000")
print("=" * 80)


MODEL COMPARISON: ALL 5 MODELS

Note: Detailed metrics, hyperparameters, and model artifacts are now
tracked in MLflow. View the dashboard with: mlflow ui

Model Performance (RMSE on original scale, EUR):
  1. Linear Regression:              €24445.42
  2. XGBoost (basic):                €12301.53
  3. Random Forest:                  €12309.59
  4. XGBoost (tuned, log-normal):    €12336.02
  5. Actuarial XGBoost (gamma):      €12113.84

Model Performance (MAPE):
  1. Linear Regression:              59.66%
  2. XGBoost (basic):                40.94%
  3. Random Forest:                  40.20%
  4. XGBoost (tuned, log-normal):    40.62%
  5. Actuarial XGBoost (gamma):      50.69%

BEST MODEL SUMMARY

Best Model: Actuarial XGBoost
  RMSE: €12113.84
  MAPE: 50.69%
  Average Claim Value: €80826.68

DETAILED PREDICTIONS ON TEST SET (First 20 claims)

Showing actual claim values vs predictions from all 5 models (in EUR):



,Actual,Linear Regression,XGBoost (Basic),Random Forest,XGBoost (Tuned),Actuarial XGBoost
0,27276.0,20660.90,14925.750000,14616.00,14702.480469,20657.580078
1,142504.0,173081.01,137813.781250,138646.77,136330.343750,136552.937500
2,119619.0,85110.61,101554.562500,102209.49,100336.656250,101294.882812
3,157104.0,177725.85,138285.625000,139011.01,138559.421875,138352.453125
4,86368.0,84504.83,100264.539062,99866.76,100155.492188,100456.203125
5,25232.0,22415.20,17802.400391,18232.49,17450.039062,22141.519531
6,70311.0,42082.29,59659.910156,59920.82,60059.460938,61711.738281
7,147760.0,173552.66,139915.703125,141316.52,141610.234375,140691.562500
8,126357.0,175923.04,138440.546875,138138.61,140232.031250,138823.234375
9,73317.0,43572.23,63417.281250,59866.29,62504.558594,63411.390625



MODEL PREDICTION ERRORS (First 20 claims)

Absolute Error (EUR) for each model:



,Actual,Linear Regression Error,XGBoost (Basic) Error,Random Forest Error,XGBoost (Tuned) Error,Actuarial XGBoost Error
0,27276.0,6615.10,12350.25,12660.00,12573.52,6618.42
1,142504.0,30577.01,4690.22,3857.23,6173.66,5951.06
2,119619.0,34508.39,18064.44,17409.51,19282.34,18324.12
3,157104.0,20621.85,18818.38,18092.99,18544.58,18751.55
4,86368.0,1863.17,13896.54,13498.76,13787.49,14088.20
5,25232.0,2816.80,7429.60,6999.51,7781.96,3090.48
6,70311.0,28228.71,10651.09,10390.18,10251.54,8599.25
7,147760.0,25792.66,7844.30,6443.48,6149.77,7068.44
8,126357.0,49566.04,12083.55,11781.61,13875.03,12466.23
9,73317.0,29744.77,9899.71,13450.71,10812.45,9905.61



MODEL ERROR STATISTICS




,Model,Mean Absolute Error (EUR),Median Absolute Error (EUR),Max Error (EUR)
0,Linear Regression,19792.70,17593.86,140150.20
1,XGBoost (Basic),9883.20,9447.66,145093.59
2,Random Forest,9855.08,9508.80,144340.07
3,XGBoost (Tuned),9910.33,9538.71,145330.92
4,Actuarial XGBoost,9739.74,9439.67,140867.57



TO VIEW MLFLOW DASHBOARD:
   Run in terminal: mlflow ui
   Then open: http://localhost:5000
